# GAMMA vs baselines — GPU training on Colab

Runs the **honest rebuild** (`src/modeling`) on a GPU: real models, temporal splits,
train-only scaling, masked quantile loss, naive floors, multi-seed CIs, Diebold-Mariano
tests, GAMMA ablations, and all journal artifacts.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is plenty).

You need the cleaned panel `air_quality_clean.parquet` (~24 MB). Cell 4 will mount Google
Drive and look for it, or fall back to a manual upload. With the default cell-6 settings
(`--stride-train 3 --num-workers 4`, 5 seeds + ablations) a T4 run is roughly **45-90 min**.
Results are written only at the end, so let cell 6 finish before downloading.

In [ ]:
# 1. Confirm GPU
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime -> Change runtime type -> GPU, then rerun.')

In [ ]:
# 2. Clone the repo (the modeling code lives under src/modeling)
REPO = 'https://github.com/Nafis878/Air-Quality-Bangladesh.git'
import os
if not os.path.isdir('Air-Quality-Bangladesh'):
    !git clone {REPO}
%cd Air-Quality-Bangladesh
!git pull --quiet
!ls src/modeling

In [ ]:
# 3. Install the few deps Colab is missing (keep Colab's CUDA torch as-is)
!pip install -q pyyaml pyarrow scipy scikit-learn seaborn

In [ ]:
# 4. Provide the cleaned panel: data/processed/air_quality_clean.parquet
import os, shutil
os.makedirs('data/processed', exist_ok=True)
dst = 'data/processed/air_quality_clean.parquet'
if not os.path.exists(dst):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        # put the file anywhere in your Drive named air_quality_clean.parquet
        import glob
        hits = glob.glob('/content/drive/MyDrive/**/air_quality_clean.parquet', recursive=True)
        if hits:
            shutil.copy(hits[0], dst); print('copied from Drive:', hits[0])
    except Exception as e:
        print('Drive mount skipped:', e)
if not os.path.exists(dst):
    from google.colab import files
    print('Upload air_quality_clean.parquet (from your local data/processed/):')
    up = files.upload()
    name = next(iter(up))
    shutil.move(name, dst)
import pandas as pd
print('panel rows:', len(pd.read_parquet(dst)))

In [ ]:
# 5. (optional) sanity: run the unit tests
!python -m pytest tests/test_modeling.py -q

In [ ]:
# 6. FULL STUDY on GPU: all 11 models x 5 seeds + ablations + naive floors.
#    --num-workers 4 keeps the GPU fed; --stride-train 3 samples a window every 3h
#    (~150k train windows/baseline, still plenty for 5-seed CIs). ~45-90 min on a T4.
#    For a faster first look: --stride-train 6 --seeds 0,1,2  (~25-35 min).
#    Lower --batch-size if you hit OOM.
!python -m src.modeling.run --config configs/modeling.yaml \
    --device cuda --stride-train 3 --seeds 0,1,2,3,4 \
    --batch-size 512 --num-workers 4 --ablations

In [ ]:
# 7. Inspect the honest verdict, then download everything
print(open('results/methods_results.md', encoding='utf-8').read())

In [ ]:
# 8. Zip + download results (metrics.json, tables, figures, report)
import shutil
shutil.make_archive('gamma_results', 'zip', 'results')
from google.colab import files
files.download('gamma_results.zip')